In [12]:
# !pip install fusion_solar_py -q

In [13]:
# from kaggle_secrets import UserSecretsClient

# user_secrets = UserSecretsClient()
# FUSION_SOLAR_CLIENT_PASSWORD = user_secrets.get_secret("FUSION_SOLAR_CLIENT_PASSWORD")
# FUSION_SOLAR_CLIENT_USERNAME = user_secrets.get_secret("FUSION_SOLAR_CLIENT_USERNAME")
# LAT = float(user_secrets.get_secret("LAT"))
# LON = float(user_secrets.get_secret("LON"))

In [14]:
import os

from dotenv import load_dotenv

load_dotenv()
FUSION_SOLAR_CLIENT_PASSWORD = os.environ.get("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = os.environ.get("FUSION_SOLAR_CLIENT_USERNAME")
LAT = float(os.environ.get("LAT"))
LON = float(os.environ.get("LON"))

In [15]:
import logging

import numpy as np

from energymanagementrl.fusion_solar_connector import *
from energymanagementrl.production_forecast import *
from energymanagementrl.rl import extract_values_gen, EnergyManagementSystem
from stable_baselines3 import DQN

In [16]:
# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Create a file handler for logging to a file
file_handler = logging.FileHandler('.log')
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Create a console handler for logging to the screen
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))

# Add both handlers to the logger

for handler in logger.handlers[:]:
    logger.removeHandler(handler)
logger.addHandler(file_handler)
logger.addHandler(console_handler)



In [17]:
panel_model = PanelModel(pdc0=0.42, temp_model_a=-3.56, temp_model_b=-0.075, delta_t=3, gamma_pdc=-0.004)
num_panels = 14
arrays = [ArrayConfig(name='sud_east', panel_model=panel_model, num_panels=num_panels, tilt_angle=25, azimuth=110),
          ArrayConfig(name='nord_west', panel_model=panel_model, num_panels=num_panels, tilt_angle=18, azimuth=290)]

_plant_config = PlantConfig(
    latitude=LAT, longitude=LON, timezone='Europe/Rome', inverter_pdc0=6, arrays=arrays
)
_production_forecaster = EnergyPredictionSystem(plant_config=_plant_config, open_meteo_client=OpenMeteoClient())

In [18]:
_client = FusionSolarClientParsed(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                  huawei_subdomain="uni004eu5")
periodic_task = PeriodicTask(_client.keep_alive)
periodic_task.start()
_plant_id = _client.get_plant_ids()[0]
battery_id = _client.get_battery_ids(_plant_id)[0]

Periodic task started.


In [19]:
_model = DQN.load(
    # '../logs/ppo_inverter.0.1.b',
    # "../logs/ppo_inverter.1.4.b",
    # "../logs/ppo_inverter.1.2.b",
    # "../logs/dqn_2.0_0.03_400_b",
    # '../logs/dqn_2.0_0.02_400_l',
    '../logs/dqn_1.0_0.05_200_b'
)

In [20]:
system = EnergyManagementSystem(client=_client, plant_id=_plant_id, battery_id=battery_id,
                                production_forecaster=_production_forecaster, model=_model, )

In [ ]:
system.control_loop(active=False)

2024-12-24 08:01:54,200 - WARNING - Battery Mode: MAXIMUM_SELF_CONSUMPTION


In [ ]:
state = system.get_system_state()
obs = np.array(list(extract_values_gen(state)))
# obs[-3]=200
# obs[-4]=0
# obs[-2]=200
# obs[-5] = 8800

if len(obs) == 55:
    action, _ = _model.predict(obs / 1000)
    print(int(action))
state